### CSV File Generation

In [35]:
import pandas as pd

df = pd.read_csv("../dataset/labels.csv")

train_df = df[df["drc"].notna()]
test_df = df[df["drc"].isna()]

train_df.to_csv("../dataset/train_labels.csv", index=False)
test_df.to_csv("../dataset/test_images.csv", index=False)

print("Files created")

Files created


In [36]:
import os
import pandas as pd

train = pd.read_csv("../dataset/train_labels.csv")

missing = []

for img in train["image"]:
    if not os.path.exists("../dataset/images/" + img):
        missing.append(img)

print("Missing images:", missing)

Missing images: []


### install and import 

In [1]:
!pip install tensorflow opencv-python pandas numpy scikit-learn

### import library

In [2]:
import pandas as pd
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split

### Load dataset

In [3]:
import pandas as pd
import os

image_folder = "../dataset/images"
csv_path = "../dataset/train_labels.csv"

df = pd.read_csv(csv_path)

print(df.head())
print("Total rows in CSV:", len(df))

          image   drc
0  IMG_4070.JPG  62.0
1  IMG_4071.JPG  62.0
2  IMG_4072.JPG  62.0
3  IMG_4073.JPG  62.0
4  IMG_4074.JPG  62.0
Total rows in CSV: 191


### VGG16 Model Code

In [4]:
import pandas as pd
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model

# Paths
image_folder = "../dataset/images"
csv_path = "../dataset/train_labels.csv"

# Load labels
df = pd.read_csv(csv_path)

images = []
labels = []

# Load images
for index,row in df.iterrows():
    
    img_path = os.path.join(image_folder, row["image"])
    
    img = cv2.imread(img_path)
    img = cv2.resize(img,(224,224))
    
    images.append(img)
    labels.append(row["drc"])

X = np.array(images)/255.0
y = np.array(labels)

# Train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Load VGG16
base_model = VGG16(weights='imagenet',include_top=False,input_shape=(224,224,3))

x = base_model.output
x = Flatten()(x)
x = Dense(128,activation='relu')(x)
output = Dense(1)(x)

model = Model(inputs=base_model.input,outputs=output)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable=False

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test,y_test)
)

# Evaluate
loss,mae = model.evaluate(X_test,y_test)

print("VGG16 Test MAE:",mae)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 44s 9s/step - loss: 2100.4304 - mae: 37.7252 - val_loss: 861.9272 - val_mae: 26.1868
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 40s 8s/step - loss: 1039.0934 - mae: 28.0171 - val_loss: 698.8483 - val_mae: 23.4021
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 40s 8s/step - loss: 246.3344 - mae: 12.7042 - val_loss: 264.1173 - val_mae: 14.2274
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 40s 8s/step - loss: 439.2748 - mae: 19.0866 - val_loss: 214.1792 - val_mae: 12.7478
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 39s 8s/step - loss: 147.9027 - mae: 10.0953 - val_loss: 216.3613 - val_mae: 11.5785
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 40s 8s/step - loss: 196.4586 - mae: 10.9271 - val_loss: 213.6371 - val_mae: 11.6291
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 41s 8s/step - loss: 93.6278 - mae: 7.1607 - val_loss: 92.7991 - val_mae: 8.2341
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 40s 8s/step - loss: 105.8332 - mae: 9.0057 - val_loss: 92.5922 - val_mae: 8.2157
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 41s 8s/st

### EfficientNet Model Code

In [5]:
import pandas as pd
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

image_folder = "../dataset/images"
csv_path = "../dataset/train_labels.csv"

df = pd.read_csv(csv_path)

images = []
labels = []

for index,row in df.iterrows():
    
    img_path = os.path.join(image_folder,row["image"])
    
    img = cv2.imread(img_path)
    img = cv2.resize(img,(224,224))
    
    images.append(img)
    labels.append(row["drc"])

X = np.array(images)/255.0
y = np.array(labels)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation='relu')(x)
output = Dense(1)(x)

model = Model(inputs=base_model.input,outputs=output)

for layer in base_model.layers:
    layer.trainable=False

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test,y_test)
)

loss,mae = model.evaluate(X_test,y_test)

print("EfficientNet Test MAE:",mae)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 44s 4s/step - loss: 4903.5742 - mae: 68.9753 - val_loss: 3989.3645 - val_mae: 62.2424
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - loss: 4094.4158 - mae: 62.8075 - val_loss: 3199.9246 - val_mae: 55.5397
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - loss: 3283.2661 - mae: 55.9885 - val_loss: 2425.5161 - val_mae: 48.0649
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - loss: 2474.7146 - mae: 48.2507 - val_loss: 1720.7950 - val_mae: 40.0686
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - loss: 1794.0306 - mae: 40.4898 - val_loss: 1119.6117 - val_mae: 31.6906
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - loss: 1207.3972 - mae: 32.3866 - val_loss: 654.6528 - val_mae: 23.4134
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - loss: 721.5981 - mae: 24.4442 - val_loss: 338.1599 - val_mae: 16.2079
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - loss: 407.2845 - mae: 17.7153 - val_loss: 167.0861 - val_mae: 11.5949
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 

### model saving

In [6]:
model.save("../models/vgg16_drc_model.keras")

### Load Saved Model

In [7]:
from tensorflow.keras.models import load_model

model = load_model("../models/vgg16_drc_model.keras")

print("Model loaded successfully")

Model loaded successfully


### Load the Unlabeled Test Images(Test_dataset)

In [8]:
import pandas as pd
import cv2
import numpy as np
import os

test_df = pd.read_csv("../dataset/test_images.csv")

image_folder = "../dataset/images"

print(test_df.head())

          image  drc
0  IMG_4039.JPG  NaN
1  IMG_4040.JPG  NaN
2  IMG_4041.JPG  NaN
3  IMG_4042.JPG  NaN
4  IMG_4043.JPG  NaN


### Predict DRC Values

In [11]:
predictions = []

for img_name in test_df["image"]:

    img_path = os.path.join(image_folder, img_name)

    img = cv2.imread(img_path)

    # check if image loaded
    if img is None:
        print("Image not found or cannot load:", img_path)
        continue

    img = cv2.resize(img, (224,224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img)

    predictions.append(pred[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

###  Predictions Into the DataFrame

In [15]:
predictions = []
image_names = []

for img_name in test_df["image"]:

    img_path = os.path.join(image_folder, img_name)

    img = cv2.imread(img_path)

    if img is None:
        print("Skipped:", img_name)
        continue

    img = cv2.resize(img, (224,224))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img, verbose=0)

    predictions.append(pred[0][0])
    image_names.append(img_name)

prediction_df = pd.DataFrame({
    "image": image_names,
    "predicted_drc": predictions
})

prediction_df.head()

Skipped: Rubber_factory


,image,predicted_drc
0,IMG_4039.JPG,73.370247
1,IMG_4040.JPG,73.312714
2,IMG_4041.JPG,73.269615
3,IMG_4042.JPG,73.261505
4,IMG_4043.JPG,73.149689


### Save Predections

In [ ]:
prediction_df.to_csv("../model_outputs/drc_predictions.csv", index=False)

print("Predictions saved successfully")

### Augmentation code

#### Refer Next NoteBook